# Querying the Replicated MPC Database

#### How to run the [replicated-table sample queries](https://docs.minorplanetcenter.net/mpc-ops-docs/data-and-services/replicated-tables-queries/) from Python.

The MPC makes its PostgreSQL database of observations and orbits available for replication through the
[SBN](https://sbnmpc.astro.umd.edu/MPC_database/statusDB.shtml). This notebook shows how to connect to such a
replica from Python and run a selection of the documented sample queries, returning the results as
[pandas](https://pandas.pydata.org/) DataFrames.

> **This notebook requires access to your own replicated copy of the MPC database.**
>
> The MPC does not host a public SQL endpoint: you obtain the data by setting up replication from the SBN
> (see the [introduction](https://docs.minorplanetcenter.net/mpc-ops-docs/data-and-services/replicated-tables-intro/)).
> The code cells below will only run once you point them at a database you can reach; they are therefore shown
> without stored output. Fill in your own connection details in the *Connect* section.

## 1. Setup

Install the required packages (once):

```bash
pip install "psycopg[binary]" sqlalchemy pandas
```

Then import them:

In [ ]:
import os

import pandas as pd
from sqlalchemy import create_engine, text

## 2. Connect

Connection details are specific to *your* replica. Here we read them from environment variables so that no
credentials are written into the notebook; set `MPC_DB_HOST`, `MPC_DB_NAME`, `MPC_DB_USER` and
`MPC_DB_PASSWORD` in your shell before launching Jupyter (or edit the defaults below).

In [ ]:
host = os.environ.get("MPC_DB_HOST", "localhost")
name = os.environ.get("MPC_DB_NAME", "mpc_sbn")
user = os.environ.get("MPC_DB_USER", "mpc_read")
password = os.environ.get("MPC_DB_PASSWORD", "")

engine = create_engine(f"postgresql+psycopg://{user}:{password}@{host}/{name}")


def run(sql, **params):
    """Run a SQL query and return the result as a pandas DataFrame.

    Pass query parameters as keyword arguments and reference them in the SQL
    with the :name style, e.g. run("... WHERE provid = :desig", desig="2010 HL23").
    """
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params or None)

## 3. Identifications and designations

All the designations (primary and secondary) associated with an object live in `current_identifications`.
Here we retrieve every secondary designation for a known primary designation:

In [ ]:
run(
    """
    SELECT unpacked_primary_provisional_designation,
           unpacked_secondary_provisional_designation
    FROM current_identifications
    WHERE unpacked_primary_provisional_designation = :desig;
    """,
    desig="2015 AC2",
)

If you only know *a* designation (not necessarily the primary one), resolve the primary first with a
sub-query, then fetch the number and name:

In [ ]:
run(
    """
    SELECT ni.permid, ni.iau_name, ni.naming_credit
    FROM numbered_identifications ni
    WHERE ni.unpacked_primary_provisional_designation = (
        SELECT unpacked_primary_provisional_designation
        FROM current_identifications
        WHERE unpacked_secondary_provisional_designation = :desig
    );
    """,
    desig="2010 HL23",
)

## 4. Observations

Observations are in `obs_sbn`. Remember to bound queries by `status` (and, for time-based queries, by an
`obstime` date range) — see the [performance notes](https://docs.minorplanetcenter.net/mpc-ops-docs/data-and-services/replicated-tables-queries/#conventions-and-performance-notes).
This returns the 80-column records for a numbered object:

In [ ]:
run(
    """
    SELECT obs80
    FROM obs_sbn
    WHERE permid = :num
      AND status IN ('P', 'p');
    """,
    num="123456",
)

A more analytical example — the number of published observations per observatory in a date window:

In [ ]:
run(
    """
    SELECT stn, count(*) AS n_obs
    FROM obs_sbn
    WHERE status IN ('P', 'p')
      AND obstime::date BETWEEN :start AND :end
    GROUP BY stn
    ORDER BY n_obs DESC
    LIMIT 20;
    """,
    start="2024-01-01", end="2024-01-31",
)

## 5. Orbits

`mpc_orbits` holds a fitted orbit for every designated object for which one could be computed, with the
elements available both as individual columns and as a complete MPC-ORB JSON document (`mpc_orb_jsonb`).

In [ ]:
run(
    """
    SELECT unpacked_primary_provisional_designation, permid,
           epoch_mjd, a, e, i, q, node, argperi, h,
           u_param, earth_moid, is_pha, orbit_type_int
    FROM mpc_orbits
    WHERE unpacked_primary_provisional_designation = :desig;
    """,
    desig="2015 AC2",
)

The `orbit_type_int` column classifies each orbit; the integer-to-class mapping is on the
[Orbit Type Definition](https://docs.minorplanetcenter.net/mpc-ops-docs/orbits/orbit-types/) page.
For example, all Apollo-type objects (`orbit_type_int = 2`), brightest first:

In [ ]:
run(
    """
    SELECT unpacked_primary_provisional_designation, a, e, i, h
    FROM mpc_orbits
    WHERE orbit_type_int = 2
    ORDER BY h ASC
    LIMIT 25;
    """,
)

## 6. Observatory codes

`obscodes` records every observatory code assigned by the MPC, including the geocentric parallax constants
`rhocosphi` / `rhosinphi` and the east `longitude`.

In [ ]:
run(
    """
    SELECT obscode, name, longitude, latitude, rhocosphi, rhosinphi, observations_type
    FROM obscodes
    WHERE obscode = :code;
    """,
    code="703",
)

## Summary

You have connected to a replicated MPC database and run queries against the identifications, observations,
orbits and observatory-code tables, returning each result as a pandas DataFrame.

Next steps:

- Browse the full [sample-query cookbook](https://docs.minorplanetcenter.net/mpc-ops-docs/data-and-services/replicated-tables-queries/) for many more examples (NEOCP, the ITF, alterations, aggregates, ...).
- Consult the [table schema](https://docs.minorplanetcenter.net/mpc-ops-docs/data-and-services/replicated-tables-schema/) for the columns available in each table.
- Review the [introduction](https://docs.minorplanetcenter.net/mpc-ops-docs/data-and-services/replicated-tables-intro/) for guidance on indexes and keeping your replica up to date.